# L14 — Transformers and Autoregressive Language Models

| Section | Content |
|---|---|
| 1 | Tokenization — subword vocabulary, BPE |
| 2 | Autoregressive language modeling — chain rule decomposition |
| 3 | Token embeddings — discrete IDs to dense vectors |
| 4 | Training and generation — negative log-likelihood, temperature, top-k |
| 5 | Single-head attention — Q, K, V projections, softmax weighting |
| 6 | Causal masking — enforcing autoregressive dependency |
| 7 | Multi-head attention — parallel heads, concatenation, projection |
| 8 | Transformer block — attention + MLP + residuals + normalization |
| 9 | Computational complexity — O(T²) attention, memory footprint |

## 1. Tokenization

A transformer is a numerical model — it requires sequences of integers, not raw text. Tokenization converts a string into a sequence of **token IDs**, where each ID indexes a fixed **vocabulary** $V$.

### Why not characters or words?

| Granularity | Problem |
|---|---|
| Character | Sequences become extremely long; each step captures too little meaning |
| Word | Rare/compound words (e.g. *internationalization*) appear too infrequently to learn well; vocabulary explodes across languages |
| **Subword** | Balances coverage and frequency — common words are single tokens, rare words are split into known pieces |

**Example:** `unhappiness` → `[un, happiness]` or `[un, h, apiness]` depending on the vocabulary. The model has seen `un-` as a prefix and `happiness` as a word — it can compose their meanings without needing to observe `unhappiness` directly.

### Byte Pair Encoding (BPE)

BPE is the dominant tokenization algorithm:
1. Start with a character-level vocabulary.
2. Greedily find the most frequent adjacent pair of tokens in the training corpus.
3. Merge that pair into a new single token and add it to the vocabulary.
4. Repeat until the vocabulary reaches size $|V|$.

Typical $|V|$: on the order of 100,000–250,000 tokens. Every model has its own tokenizer — the vocabulary is part of the model specification.

**Notation from here on:** a piece of text is a sequence $x_1, x_2, \ldots, x_T$ where each $x_t \in V$ is a token ID. The sequence length $T$ varies per example.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter


def bpe_step(vocab_counts):
    """One BPE merge step: find the most frequent adjacent pair and merge it."""
    pair_counts = Counter()
    for word, freq in vocab_counts.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pair_counts[(symbols[i], symbols[i + 1])] += freq
    if not pair_counts:
        return vocab_counts, None
    best_pair = max(pair_counts, key=pair_counts.get)
    merged = best_pair[0] + best_pair[1]
    new_vocab = {}
    for word, freq in vocab_counts.items():
        new_word = word.replace(' '.join(best_pair), merged)
        new_vocab[new_word] = freq
    return new_vocab, best_pair


# Small corpus — token-level BPE demo
corpus = {
    'l o w </w>': 5,
    'l o w e r </w>': 2,
    'n e w e s t </w>': 6,
    'w i d e s t </w>': 3,
}

print('BPE merges on a toy corpus:')
print(f'  Start: {list(corpus.keys())}')
merges = []
vocab = dict(corpus)
for step in range(8):
    vocab, pair = bpe_step(vocab)
    if pair is None:
        break
    merges.append(pair)
    print(f'  Merge {step+1}: {pair[0]} + {pair[1]} → {pair[0]+pair[1]}')

print(f'\nFinal token forms:')
for word, freq in vocab.items():
    print(f'  {word}  (freq={freq})')

# Visualize merge frequency selection
initial_corpus = {
    'l o w </w>': 5,
    'l o w e r </w>': 2,
    'n e w e s t </w>': 6,
    'w i d e s t </w>': 3,
}
pair_counts = Counter()
for word, freq in initial_corpus.items():
    symbols = word.split()
    for i in range(len(symbols) - 1):
        pair_counts[(symbols[i], symbols[i + 1])] += freq

top_pairs = pair_counts.most_common(8)
labels = [f'{a}+{b}' for (a, b), _ in top_pairs]
counts = [c for _, c in top_pairs]

fig, ax = plt.subplots(figsize=(9, 3.5))
bars = ax.bar(labels, counts, color='steelblue', alpha=0.8)
bars[0].set_color('red')
ax.set_xlabel('Adjacent pair')
ax.set_ylabel('Frequency in corpus')
ax.set_title('BPE Step 1 — Most Frequent Pair Selected for Merge (red)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Autoregressive Language Modeling

Given a vocabulary $V$, a sequence of length $T$ has $|V|^T$ possible realizations. Assigning one probability to each realization requires a table with $|V|^T$ entries — completely intractable for $|V| \approx 10^5$ and $T \approx 10^3$.

### Chain rule decomposition

The probability of any sequence factors exactly:

$$p(x_1, x_2, \ldots, x_T) = p(x_1) \cdot p(x_2 \mid x_1) \cdot p(x_3 \mid x_1, x_2) \cdots p(x_T \mid x_1, \ldots, x_{T-1})$$

Each conditional $p(x_t \mid x_1, \ldots, x_{t-1})$ is a distribution over $|V|$ choices — a softmax vector, not a table. This is tractable.

### Parameterization

A transformer $f_\theta$ takes the sequence seen so far and outputs a logit vector in $\mathbb{R}^{|V|}$ at each position:

$$p_\theta(x_t \mid x_1, \ldots, x_{t-1}) = \text{softmax}\bigl(f_\theta(x_0, x_1, \ldots, x_{t-1})\bigr)$$

where $x_0$ is a special beginning-of-sequence token. The subscript denotes which time step's output is used — the output at position $t-1$ gives the distribution over $x_t$.

**Row vector convention (matches implementation):** embeddings and intermediate representations are row vectors $\in \mathbb{R}^{1 \times d}$. Matrix multiplications are applied on the right: $h W$ rather than $W^\top h$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def softmax(x):
    x = x - np.max(x)
    e = np.exp(x)
    return e / e.sum()


# Illustrate chain rule factorization on a tiny 3-token vocabulary
np.random.seed(7)
vocab = ['cat', 'sat', 'mat']
V = len(vocab)

# Toy conditional distributions (hand-set for illustration)
# p(x1)  — unconditional first token
logits_1 = np.array([2.0, 0.5, -0.5])
p1 = softmax(logits_1)

# p(x2 | x1='cat')  — after seeing 'cat'
logits_2_given_cat = np.array([-0.5, 2.5, 0.3])
p2_given_cat = softmax(logits_2_given_cat)

# p(x3 | x1='cat', x2='sat')  — after 'cat sat'
logits_3_given_catsat = np.array([0.1, -1.0, 3.0])
p3_given_catsat = softmax(logits_3_given_catsat)

seq = ('cat', 'sat', 'mat')
joint = p1[0] * p2_given_cat[1] * p3_given_catsat[2]

print('Chain rule factorization example:')
print(f'  Vocabulary: {vocab}')
print(f'  p(cat)              = {p1[0]:.4f}')
print(f'  p(sat | cat)        = {p2_given_cat[1]:.4f}')
print(f'  p(mat | cat, sat)   = {p3_given_catsat[2]:.4f}')
print(f'  p("cat sat mat")    = {joint:.6f}  [product of conditionals]')

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
titles = ['p(x₁)', 'p(x₂ | x₁="cat")', 'p(x₃ | x₁="cat", x₂="sat")']
dists  = [p1, p2_given_cat, p3_given_catsat]
for ax, dist, title in zip(axes, dists, titles):
    bars = ax.bar(vocab, dist, color='steelblue', alpha=0.8)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Probability')
    ax.set_title(title)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, dist):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.02, f'{val:.2f}',
                ha='center', fontsize=9)

plt.suptitle('Autoregressive Chain Rule: Each Step is a Softmax over V', fontsize=11)
plt.tight_layout()
plt.show()

## 3. Token Embeddings

Token IDs are discrete integers — they have no geometric meaning. Before feeding tokens into any neural network computation, each ID is mapped to a learned dense vector called an **embedding**.

### Embedding matrix

Define an embedding matrix $E \in \mathbb{R}^{|V| \times d}$. The embedding for token $x_t \in V$ is the $x_t$-th row:

$$e_t = E[x_t, :] \in \mathbb{R}^d$$

Equivalently, if $\text{onehot}(x_t) \in \{0,1\}^{|V|}$ is the indicator vector, then $e_t = \text{onehot}(x_t) \cdot E$. Both views are identical; in practice, the lookup is implemented as an index operation.

The full sequence $x_0, x_1, \ldots, x_T$ becomes a matrix of embeddings:

$$\mathbf{E}_{\text{seq}} = \begin{bmatrix} e_0 \\ e_1 \\ \vdots \\ e_T \end{bmatrix} \in \mathbb{R}^{(T+1) \times d}$$

This matrix is the input to the first transformer layer. The embedding parameters $E$ are trained end-to-end along with all other model parameters.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


np.random.seed(0)
V_size = 10   # toy vocabulary size
d = 8         # embedding dimension

# Embedding matrix: |V| rows, d columns
E = np.random.randn(V_size, d) * 0.3

# A short token sequence: IDs [0, 3, 7, 2]
token_ids = np.array([0, 3, 7, 2])
T_seq = len(token_ids)

# Lookup embeddings for each token
E_seq = E[token_ids, :]   # shape (T, d)

print(f'Vocabulary size: {V_size}')
print(f'Embedding dim d: {d}')
print(f'Token sequence (IDs): {token_ids}')
print(f'Embedding matrix E shape: {E.shape}')
print(f'Sequence embedding matrix shape: {E_seq.shape}  (one row per token)')

# Verify: onehot @ E equals direct row lookup
onehot = np.zeros((T_seq, V_size))
onehot[np.arange(T_seq), token_ids] = 1.0
E_via_matmul = onehot @ E
assert np.allclose(E_seq, E_via_matmul), 'Mismatch'
print('Row-lookup and onehot @ E give identical results.')

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

im0 = axes[0].imshow(E, aspect='auto', cmap='RdBu_r')
axes[0].set_title(f'Embedding matrix E  ({V_size}×{d})')
axes[0].set_xlabel('Embedding dimension'); axes[0].set_ylabel('Token ID')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(onehot, aspect='auto', cmap='Blues')
axes[1].set_title('One-hot input  (T×|V|)')
axes[1].set_xlabel('Token ID'); axes[1].set_ylabel('Sequence position')
axes[1].set_yticks(range(T_seq)); axes[1].set_yticklabels([f'pos {i}' for i in range(T_seq)])
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(E_seq, aspect='auto', cmap='RdBu_r')
axes[2].set_title('Sequence embeddings E_seq  (T×d)')
axes[2].set_xlabel('Embedding dimension'); axes[2].set_ylabel('Sequence position')
axes[2].set_yticks(range(T_seq)); axes[2].set_yticklabels([f'id={i}' for i in token_ids])
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.suptitle('Token Embeddings: Discrete IDs → Dense Vectors', fontsize=11)
plt.tight_layout()
plt.show()

## 4. Training and Generation

### Training — negative log-likelihood

Given a training sequence $x_1, \ldots, x_T$, the loss is:

$$\mathcal{L}(\theta) = -\sum_{t=1}^{T} \log p_\theta(x_t \mid x_0, \ldots, x_{t-1})$$

Each term is the log of the entry indexed by $x_t$ in the softmax output at position $t-1$. Minimizing this is equivalent to maximizing the probability the model assigns to the observed sequence.

The transformer computes all conditional distributions in a single forward pass over the full sequence (enabled by causal masking, Section 6).

### Generation — autoregressive decoding

Starting from $x_0$ (the BOS token):

1. Run the transformer on $(x_0)$; sample $x_1 \sim \text{softmax}(f_\theta(x_0))$.
2. Run on $(x_0, x_1)$; sample $x_2 \sim \text{softmax}(f_\theta(x_0, x_1))$.
3. Repeat until an end-of-sequence token is generated or a length limit is reached.

When a prompt $(x_1, \ldots, x_K)$ is given, generation starts from position $K+1$.

### Temperature scaling

Before the softmax, the logit vector $u \in \mathbb{R}^{|V|}$ can be rescaled by temperature $\tau > 0$:

$$p_\tau(x) = \text{softmax}(u / \tau)$$

- $\tau \to 0$: distribution sharpens — mass concentrates on the highest-logit token (greedy/deterministic decoding).
- $\tau = 1$: original model distribution.
- $\tau > 1$: distribution flattens — more randomness, longer-tail sampling.

**Top-k sampling:** zero out all but the top-$k$ logit entries, renormalize, then sample. Prevents sampling from very rare tokens without changing the ordering.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def softmax_temp(logits, tau=1.0):
    u = logits / tau
    u = u - u.max()
    e = np.exp(u)
    return e / e.sum()


def topk_sample(logits, k):
    top_indices = np.argsort(logits)[::-1][:k]
    mask = np.full_like(logits, -np.inf)
    mask[top_indices] = logits[top_indices]
    return softmax_temp(mask, tau=1.0)


# 10-token vocabulary logits
np.random.seed(12)
logits = np.array([2.0, 1.0, -1.0, 0.5, -0.3, 1.8, -2.0, 0.2, 0.7, -0.5])
vocab_labels = [f't{i}' for i in range(len(logits))]

temps = [0.1, 0.5, 1.0, 2.0]
dists = [softmax_temp(logits, t) for t in temps]
topk_dist = topk_sample(logits, k=3)

fig, axes = plt.subplots(1, 5, figsize=(16, 3.5), sharey=True)
colors = plt.cm.coolwarm(np.linspace(0, 1, 4))
for ax, dist, tau, col in zip(axes[:4], dists, temps, colors):
    ax.bar(vocab_labels, dist, color=col, alpha=0.85)
    ax.set_title(f'τ = {tau}')
    ax.set_xlabel('Token')
    ax.tick_params(axis='x', labelsize=7)
    ax.grid(axis='y', alpha=0.3)

axes[4].bar(vocab_labels, topk_dist, color='green', alpha=0.8)
axes[4].set_title('Top-3 sampling')
axes[4].set_xlabel('Token')
axes[4].tick_params(axis='x', labelsize=7)
axes[4].grid(axis='y', alpha=0.3)

axes[0].set_ylabel('Probability')
plt.suptitle('Temperature and Top-k Effects on Sampling Distribution', fontsize=11)
plt.tight_layout()
plt.show()

print('Effect on entropy (bits):')
for dist, tau in zip(dists, temps):
    H = -np.sum(dist * np.log2(dist + 1e-12))
    print(f'  τ={tau}: H = {H:.3f} bits')
H_topk = -np.sum(topk_dist * np.log2(topk_dist + 1e-12))
print(f'  top-3:   H = {H_topk:.3f} bits')

## 5. Single-Head Attention

The attention layer is the mechanism that allows information to flow between sequence positions. Its type signature:

- **Input:** sequence of vectors $h_1^{\text{in}}, \ldots, h_T^{\text{in}}$, each $\in \mathbb{R}^d$
- **Output:** sequence of vectors $h_1^{\text{out}}, \ldots, h_T^{\text{out}}$, each $\in \mathbb{R}^d$

### Q, K, V projections

Three learned weight matrices $W_Q, W_K \in \mathbb{R}^{d \times d_h}$ and $W_V \in \mathbb{R}^{d \times d_h}$ project each input into query, key, and value vectors:

$$q_t = h_t^{\text{in}} W_Q, \quad k_t = h_t^{\text{in}} W_K, \quad v_t = h_t^{\text{in}} W_V$$

Stacked into matrices: $Q = H^{\text{in}} W_Q$, $K = H^{\text{in}} W_K$, $V = H^{\text{in}} W_V$, all $\in \mathbb{R}^{T \times d_h}$.

### Attention weights

For position $t$, the attention weight toward position $j$ is proportional to the inner product $q_t \cdot k_j$. Collecting all positions into a matrix:

$$A = \text{softmax}_{\text{rowwise}}\!\left(\frac{Q K^\top}{\sqrt{d_h}}\right) \in \mathbb{R}^{T \times T}$$

The $\sqrt{d_h}$ scaling prevents the inner products from growing large in high dimensions, keeping the softmax numerically stable and the gradients well-behaved.

### Output

$$H^{\text{out}} = A \cdot V \in \mathbb{R}^{T \times d_h}$$

Row $t$ of $H^{\text{out}}$ is a weighted average of all value vectors, weighted by how much position $t$ attends to each other position.

**Intuition:** $q_t$ encodes what position $t$ is looking for. $k_j$ encodes what position $j$ offers. The inner product $q_t \cdot k_j$ measures relevance. High relevance → high weight → more of $v_j$ is included in the output at position $t$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def softmax_rows(X):
    X = X - X.max(axis=-1, keepdims=True)
    e = np.exp(X)
    return e / e.sum(axis=-1, keepdims=True)


def single_head_attention(H, WQ, WK, WV, mask=None):
    T, d = H.shape
    dh = WQ.shape[1]
    Q = H @ WQ            # (T, dh)
    K = H @ WK            # (T, dh)
    V = H @ WV            # (T, dh)
    scores = Q @ K.T / np.sqrt(dh)   # (T, T)
    if mask is not None:
        scores = scores + mask        # mask entries become -inf → 0 after softmax
    A = softmax_rows(scores)          # (T, T)
    H_out = A @ V                     # (T, dh)
    return H_out, A, Q, K, V


np.random.seed(42)
T = 6       # sequence length
d = 16      # input dimension
dh = 8      # head dimension

H_in = np.random.randn(T, d)   # input sequence (T, d)
WQ = np.random.randn(d, dh) / np.sqrt(d)
WK = np.random.randn(d, dh) / np.sqrt(d)
WV = np.random.randn(d, dh) / np.sqrt(d)

H_out, A, Q, K, V = single_head_attention(H_in, WQ, WK, WV)

print(f'Input H shape:   {H_in.shape}  (T={T}, d={d})')
print(f'Q, K, V shapes:  {Q.shape}  (T={T}, dh={dh})')
print(f'Attention A shape: {A.shape}  (T×T softmax weights)')
print(f'Output H_out shape: {H_out.shape}')
print(f'A rows sum to 1: {np.allclose(A.sum(axis=1), 1.0)}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

im0 = axes[0].imshow(A, cmap='Blues', vmin=0, vmax=A.max())
axes[0].set_title('Attention weights A  (T×T)')
axes[0].set_xlabel('Key position (j)'); axes[0].set_ylabel('Query position (t)')
for i in range(T):
    for j in range(T):
        axes[0].text(j, i, f'{A[i,j]:.2f}', ha='center', va='center', fontsize=7,
                     color='white' if A[i,j] > 0.3 else 'black')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(H_out, aspect='auto', cmap='RdBu_r')
axes[1].set_title('Output H_out  (T×dh)')
axes[1].set_xlabel('Head dimension'); axes[1].set_ylabel('Sequence position')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

plt.suptitle('Single-Head Attention: Full Attention (No Mask)', fontsize=11)
plt.tight_layout()
plt.show()

## 6. Causal Masking

Without modification, the attention matrix $A$ at row $t$ can attend to positions $j > t$ — future tokens. This violates the autoregressive constraint: the distribution $p(x_t \mid x_1, \ldots, x_{t-1})$ must not depend on $x_t, x_{t+1}, \ldots$.

### Masking mechanism

Before applying softmax, add $-\infty$ to every entry $(t, j)$ where $j > t$:

$$\text{scores}_{tj} \leftarrow \text{scores}_{tj} + M_{tj}, \quad M_{tj} = \begin{cases} 0 & j \leq t \\ -\infty & j > t \end{cases}$$

After softmax, $\exp(-\infty) = 0$, so those entries contribute nothing to the output.

The mask matrix $M$ is the strictly upper triangular part set to $-\infty$, zeros elsewhere. It is not learned — it is a fixed structural constraint.

**Key result:** with the causal mask, the entire sequence is processed in a single forward pass during training, yet each output position $t$ only uses inputs from positions $\leq t$. This is what makes parallel training possible while maintaining autoregressive semantics.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def causal_mask(T):
    """Return additive causal mask: 0 on and below diagonal, -inf above."""
    M = np.zeros((T, T))
    M[np.triu_indices(T, k=1)] = -np.inf
    return M


np.random.seed(42)
T = 6
d = 16
dh = 8

H_in = np.random.randn(T, d)
WQ = np.random.randn(d, dh) / np.sqrt(d)
WK = np.random.randn(d, dh) / np.sqrt(d)
WV = np.random.randn(d, dh) / np.sqrt(d)

M = causal_mask(T)
H_out_causal, A_causal, _, _, _ = single_head_attention(H_in, WQ, WK, WV, mask=M)
_, A_full, _, _, _ = single_head_attention(H_in, WQ, WK, WV, mask=None)

print(f'Causal mask (additive, -inf above diagonal):')
print(np.where(np.isinf(M), '-inf', '0  ').reshape(T, T))
print(f'\nA_causal upper triangle is zero: {np.allclose(np.triu(A_causal, k=1), 0.0)}')
print(f'A_causal rows sum to 1:          {np.allclose(A_causal.sum(axis=1), 1.0)}')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

mask_display = np.where(np.isinf(M), -1.0, 0.0)
im0 = axes[0].imshow(mask_display, cmap='Reds', vmin=-1, vmax=0)
axes[0].set_title('Causal mask M')
axes[0].set_xlabel('j'); axes[0].set_ylabel('t')
for i in range(T):
    for j in range(T):
        txt = '-∞' if j > i else '0'
        axes[0].text(j, i, txt, ha='center', va='center', fontsize=9)
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(A_full, cmap='Blues', vmin=0, vmax=A_full.max())
axes[1].set_title('Full attention A (no mask)')
axes[1].set_xlabel('j'); axes[1].set_ylabel('t')
for i in range(T):
    for j in range(T):
        axes[1].text(j, i, f'{A_full[i,j]:.2f}', ha='center', va='center', fontsize=7,
                     color='white' if A_full[i,j] > 0.3 else 'black')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(A_causal, cmap='Blues', vmin=0, vmax=A_causal.max())
axes[2].set_title('Causal attention A (with mask)')
axes[2].set_xlabel('j'); axes[2].set_ylabel('t')
for i in range(T):
    for j in range(T):
        val = A_causal[i,j]
        axes[2].text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7,
                     color='white' if val > 0.3 else ('gray' if val == 0 else 'black'))
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.suptitle('Causal Masking: Attention Cannot Look Forward in Time', fontsize=11)
plt.tight_layout()
plt.show()

## 7. Multi-Head Attention

A single set of $W_Q, W_K, W_V$ weights allows only one type of interaction pattern across the sequence. In practice, different aspects of context are relevant simultaneously — syntactic, semantic, positional, entity-level, etc.

### Parallel heads

Run $N_h$ independent single-head attention operations in parallel. Head $i$ has its own projection matrices $W_Q^{(i)}, W_K^{(i)}, W_V^{(i)}$ and produces its own output sequence $H^{(i)} \in \mathbb{R}^{T \times d_h}$.

### Concatenation and output projection

The outputs of all heads are concatenated along the feature dimension and projected back to the original dimension $d$:

$$H^{\text{MHA}} = \text{Concat}\bigl(H^{(1)}, \ldots, H^{(N_h)}\bigr)\, W_O$$

where $\text{Concat}(\cdot) \in \mathbb{R}^{T \times N_h d_h}$ and $W_O \in \mathbb{R}^{N_h d_h \times d}$.

Typically $N_h \cdot d_h = d$, so each head operates in a $d/N_h$-dimensional subspace.

**Parameter count per MHA layer:**
- Projections: $N_h \times 3 \times d \times d_h = 3d^2$ (when $N_h d_h = d$)
- Output projection: $N_h d_h \times d = d^2$
- Total: $4d^2$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def multi_head_attention(H, WQs, WKs, WVs, WO, mask=None):
    """
    WQs, WKs, WVs: lists of (d, dh) matrices, one per head
    WO: (Nh*dh, d) output projection
    """
    Nh = len(WQs)
    head_outputs = []
    attn_maps = []
    for i in range(Nh):
        H_i, A_i, _, _, _ = single_head_attention(H, WQs[i], WKs[i], WVs[i], mask=mask)
        head_outputs.append(H_i)
        attn_maps.append(A_i)
    concat = np.concatenate(head_outputs, axis=-1)  # (T, Nh*dh)
    H_mha = concat @ WO                              # (T, d)
    return H_mha, attn_maps


np.random.seed(99)
T = 6
d = 16
Nh = 4
dh = d // Nh   # 4: each head operates in 4-dim subspace

H_in = np.random.randn(T, d)
WQs = [np.random.randn(d, dh) / np.sqrt(d) for _ in range(Nh)]
WKs = [np.random.randn(d, dh) / np.sqrt(d) for _ in range(Nh)]
WVs = [np.random.randn(d, dh) / np.sqrt(d) for _ in range(Nh)]
WO  = np.random.randn(Nh * dh, d) / np.sqrt(Nh * dh)

M = causal_mask(T)
H_mha, attn_maps = multi_head_attention(H_in, WQs, WKs, WVs, WO, mask=M)

print(f'Input H shape:       {H_in.shape}')
print(f'Heads: {Nh}, each dh={dh}')
print(f'Output H_mha shape:  {H_mha.shape}  (same as input — d preserved)')

fig, axes = plt.subplots(1, Nh + 1, figsize=(16, 3.5))

for i, (A_i, ax) in enumerate(zip(attn_maps, axes[:Nh])):
    im = ax.imshow(A_i, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'Head {i+1} attention')
    ax.set_xlabel('j'); ax.set_ylabel('t')
    plt.colorbar(im, ax=ax, fraction=0.046)

im_out = axes[Nh].imshow(H_mha, aspect='auto', cmap='RdBu_r')
axes[Nh].set_title('MHA output H_mha  (T×d)')
axes[Nh].set_xlabel('d'); axes[Nh].set_ylabel('t')
plt.colorbar(im_out, ax=axes[Nh], fraction=0.046)

plt.suptitle(f'Multi-Head Attention: {Nh} Parallel Heads + Output Projection', fontsize=11)
plt.tight_layout()
plt.show()

param_count = Nh * 3 * d * dh + Nh * dh * d
print(f'\nTotal MHA parameters: {param_count} = 4d² = 4×{d}² = {4*d*d}')

## 8. Transformer Block — Attention, MLP, Residuals, Normalization

A transformer block applies attention and MLP in sequence, with two structural additions:

### Residual connections

Following the ResNet design, each sub-layer has a skip connection:

$$H \leftarrow H + \text{MHA}(H) \qquad H \leftarrow H + \text{MLP}(H)$$

Residuals allow gradients to flow unchanged through many layers, enabling very deep networks to train.

### Normalization (RMSNorm / LayerNorm)

Applied to stabilize activations before each sub-layer. RMSNorm (common in modern architectures) normalizes each vector by its root-mean-square:

$$\text{RMSNorm}(h)_i = \frac{h_i}{\sqrt{\frac{1}{d}\sum_j h_j^2 + \varepsilon}} \cdot \gamma_i$$

where $\gamma \in \mathbb{R}^d$ is a learned scale parameter.

### MLP (feedforward network)

Applied **independently** to each position — no cross-position interaction:

$$\text{MLP}(h) = \max(0,\, h W_1 + b_1)\, W_2 + b_2$$

where $W_1 \in \mathbb{R}^{d \times d_{\text{ff}}}$, $W_2 \in \mathbb{R}^{d_{\text{ff}} \times d}$, typically $d_{\text{ff}} = 4d$.

**Division of labor:** attention fuses information across positions; MLP processes each position's representation independently with more capacity.

### Pre-norm order (modern standard)

```
h ← h + MHA(Norm(h))
h ← h + MLP(Norm(h))
```

Applying normalization before (pre-norm) rather than after (post-norm) each sub-layer produces more stable training dynamics for very deep models.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def rms_norm(H, gamma, eps=1e-6):
    rms = np.sqrt((H ** 2).mean(axis=-1, keepdims=True) + eps)
    return (H / rms) * gamma


def mlp(H, W1, b1, W2, b2):
    return np.maximum(0, H @ W1 + b1) @ W2 + b2


def transformer_block(H, WQs, WKs, WVs, WO, W1, b1, W2, b2, gamma1, gamma2, mask=None):
    # Pre-norm attention with residual
    H_norm = rms_norm(H, gamma1)
    H_mha, _ = multi_head_attention(H_norm, WQs, WKs, WVs, WO, mask=mask)
    H = H + H_mha
    # Pre-norm MLP with residual
    H_norm = rms_norm(H, gamma2)
    H = H + mlp(H_norm, W1, b1, W2, b2)
    return H


np.random.seed(5)
T = 6
d = 16
Nh = 4
dh = d // Nh
d_ff = 4 * d   # feedforward expansion

H_in = np.random.randn(T, d) * 0.5

WQs   = [np.random.randn(d, dh) / np.sqrt(d) for _ in range(Nh)]
WKs   = [np.random.randn(d, dh) / np.sqrt(d) for _ in range(Nh)]
WVs   = [np.random.randn(d, dh) / np.sqrt(d) for _ in range(Nh)]
WO    = np.random.randn(Nh * dh, d) / np.sqrt(Nh * dh)
W1    = np.random.randn(d, d_ff) / np.sqrt(d)
b1    = np.zeros(d_ff)
W2    = np.random.randn(d_ff, d) / np.sqrt(d_ff)
b2    = np.zeros(d)
gamma1 = np.ones(d)
gamma2 = np.ones(d)

M = causal_mask(T)
H_out = transformer_block(H_in, WQs, WKs, WVs, WO, W1, b1, W2, b2, gamma1, gamma2, mask=M)

print(f'Transformer block input shape:  {H_in.shape}')
print(f'Transformer block output shape: {H_out.shape}')

param_mha = Nh * 3 * d * dh + Nh * dh * d
param_mlp = d * d_ff + d_ff * d
param_norm = 2 * d
total = param_mha + param_mlp + param_norm
print(f'\nParameters per block:')
print(f'  MHA:         {param_mha:>7,}  (4d²)')
print(f'  MLP:         {param_mlp:>7,}  (8d²  when d_ff=4d)')
print(f'  Norms:       {param_norm:>7,}  (2d)')
print(f'  Total:       {total:>7,}')

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

im0 = axes[0].imshow(H_in, aspect='auto', cmap='RdBu_r')
axes[0].set_title('Input H_in  (T×d)')
axes[0].set_xlabel('d'); axes[0].set_ylabel('t')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(H_out, aspect='auto', cmap='RdBu_r')
axes[1].set_title('Output H_out  (T×d)')
axes[1].set_xlabel('d'); axes[1].set_ylabel('t')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

diff = H_out - H_in
im2 = axes[2].imshow(diff, aspect='auto', cmap='PRGn')
axes[2].set_title('Difference H_out − H_in')
axes[2].set_xlabel('d'); axes[2].set_ylabel('t')
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.suptitle('Transformer Block (Pre-Norm): Input → MHA+Residual → MLP+Residual → Output', fontsize=10)
plt.tight_layout()
plt.show()

# Stack multiple blocks and verify shapes are preserved
H = H_in.copy()
n_layers = 4
for layer in range(n_layers):
    WQs_l  = [np.random.randn(d, dh) / np.sqrt(d) for _ in range(Nh)]
    WKs_l  = [np.random.randn(d, dh) / np.sqrt(d) for _ in range(Nh)]
    WVs_l  = [np.random.randn(d, dh) / np.sqrt(d) for _ in range(Nh)]
    WO_l   = np.random.randn(Nh * dh, d) / np.sqrt(Nh * dh)
    W1_l   = np.random.randn(d, d_ff) / np.sqrt(d)
    W2_l   = np.random.randn(d_ff, d) / np.sqrt(d_ff)
    H = transformer_block(H, WQs_l, WKs_l, WVs_l, WO_l,
                           W1_l, np.zeros(d_ff), W2_l, np.zeros(d),
                           np.ones(d), np.ones(d), mask=M)
    print(f'  After layer {layer+1}: shape {H.shape}, mean={H.mean():.4f}, std={H.std():.4f}')

## 9. Computational Complexity — O(T²) Attention

### Operations in attention

The attention score matrix $QK^\top \in \mathbb{R}^{T \times T}$ has $T^2$ entries. Each entry is an inner product of $d_h$-dimensional vectors, costing $O(d_h)$ operations:

$$\text{FLOPs for } QK^\top: \quad O(T^2 \cdot d_h)$$

For a full model with $N_h$ heads and $L$ layers:

$$\text{Total attention FLOPs} \sim O(L \cdot N_h \cdot T^2 \cdot d_h) = O(L \cdot T^2 \cdot d)$$

The MLP at each layer is $O(L \cdot T \cdot d \cdot d_{\text{ff}}) = O(L \cdot T \cdot d^2)$ — linear in $T$.

**Attention dominates when $T \gg d$**, which is common for long-context tasks.

### Memory

Storing $QK^\top$ naively requires $O(T^2)$ memory per layer. For $T = 10^6$, this is $10^{12}$ entries — infeasible.

**FlashAttention** avoids materializing the full $T \times T$ matrix by computing attention in tiles that fit in GPU SRAM, streaming through the computation. This reduces memory to $O(T)$ while keeping the same mathematical result.

### Long-context implications

The $T^2$ scaling is why LLM systems compact or truncate context — the computational cost grows quadratically. Variants of attention (sparse attention, linear attention, state space models) trade expressivity for better $T$-scaling.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time


def attention_flops(T, d, Nh=1, L=1):
    dh = d // Nh
    # QK^T: T^2 * dh multiplications, Nh heads, L layers
    attn = 2 * L * Nh * T * T * dh   # 2x for multiply-add
    # AV: T^2 * dh
    attn += 2 * L * Nh * T * T * dh
    # Projections (Q,K,V,O): 4 * T * d * d
    proj = 2 * L * 4 * T * d * d
    # MLP: T * d * d_ff * 2 (two matrices), d_ff=4d
    mlp_ops = 2 * L * T * d * 4 * d * 2
    return attn, proj, mlp_ops


seq_lengths = [128, 256, 512, 1024, 2048, 4096, 8192]
d = 512
Nh = 8
L = 12

attn_costs, proj_costs, mlp_costs = [], [], []
for T in seq_lengths:
    a, p, m = attention_flops(T, d, Nh, L)
    attn_costs.append(a)
    proj_costs.append(p)
    mlp_costs.append(m)

attn_costs = np.array(attn_costs, dtype=float)
proj_costs = np.array(proj_costs, dtype=float)
mlp_costs  = np.array(mlp_costs, dtype=float)
total_costs = attn_costs + proj_costs + mlp_costs

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].loglog(seq_lengths, attn_costs / 1e9, 'o-', color='red',      ms=6, lw=2, label='Attention (O(T²d))')
axes[0].loglog(seq_lengths, mlp_costs  / 1e9, 's-', color='steelblue',ms=6, lw=2, label='MLP       (O(Td²))')
axes[0].loglog(seq_lengths, proj_costs / 1e9, '^-', color='green',    ms=6, lw=2, label='Projections (O(Td²))')
axes[0].loglog(seq_lengths, total_costs / 1e9,'D-', color='black',    ms=6, lw=1.5, ls='--', label='Total')
# Reference lines
T_arr = np.array(seq_lengths, dtype=float)
axes[0].loglog(T_arr, T_arr**2 / T_arr[-1]**2 * total_costs[-1] / 1e9,
               color='red', lw=0.8, ls=':', alpha=0.5, label='T² slope')
axes[0].set_xlabel('Sequence length T')
axes[0].set_ylabel('GFLOPs')
axes[0].set_title(f'FLOPs vs Sequence Length  (d={d}, Nh={Nh}, L={L})')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3, which='both')

# Memory: T^2 attention matrix per head per layer (in MB, float32)
mem_attn_MB = np.array([L * Nh * T * T * 4 / 1e6 for T in seq_lengths])
mem_kv_MB   = np.array([L * 2 * T * d * 4 / 1e6  for T in seq_lengths])  # K,V cache

axes[1].semilogy(seq_lengths, mem_attn_MB, 'o-', color='red',      ms=6, lw=2, label='Attn matrix (T²×Nh×L)')
axes[1].semilogy(seq_lengths, mem_kv_MB,   's-', color='steelblue',ms=6, lw=2, label='KV cache (T×d×L)')
axes[1].axhline(40 * 1024, color='gray', ls='--', lw=1, label='40 GB GPU limit')
axes[1].set_xlabel('Sequence length T')
axes[1].set_ylabel('Memory (MB)')
axes[1].set_title('Memory Footprint vs Sequence Length')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3, which='both')

plt.suptitle('Attention Complexity: O(T²) Compute and Memory', fontsize=11)
plt.tight_layout()
plt.show()

print('FLOPs breakdown at T=2048 (% of total):')
idx = seq_lengths.index(2048)
tot = total_costs[idx]
print(f'  Attention:   {100*attn_costs[idx]/tot:.1f}%')
print(f'  Projections: {100*proj_costs[idx]/tot:.1f}%')
print(f'  MLP:         {100*mlp_costs[idx]/tot:.1f}%')
print(f'\nAttn matrix memory at T=8192: {mem_attn_MB[-1]:,.0f} MB  ({mem_attn_MB[-1]/1024:.1f} GB)')
print('FlashAttention avoids materializing this matrix — keeps memory O(T) instead of O(T²).')

## Summary

| Component | Role | Key equation |
|---|---|---|
| **Tokenization (BPE)** | Text → sequence of integer IDs | Greedy merge of frequent subword pairs |
| **Chain rule decomposition** | Tractable joint probability | $p(x_{1:T}) = \prod_t p(x_t \mid x_{<t})$ |
| **Embedding matrix** | Discrete IDs → dense vectors | $e_t = E[x_t, :]$ |
| **NLL training loss** | Maximize sequence likelihood | $\mathcal{L} = -\sum_t \log p_\theta(x_t \mid x_{<t})$ |
| **Temperature / top-k** | Control generation randomness | $p_\tau = \text{softmax}(u/\tau)$ |
| **Single-head attention** | Weighted aggregation across positions | $A = \text{softmax}(QK^\top / \sqrt{d_h})$, $H^{\text{out}} = AV$ |
| **Causal mask** | No future token leakage | Add $-\infty$ to upper triangle of score matrix |
| **Multi-head attention** | Parallel diverse interaction patterns | $\text{Concat}(H^{(1)}, \ldots, H^{(N_h)}) W_O$ |
| **Transformer block** | Full layer with residuals + norm | Pre-norm: $h \leftarrow h + \text{sub-layer}(\text{Norm}(h))$ |
| **O(T²) complexity** | Attention scales quadratically in T | FLOPs $\sim T^2 d$; FlashAttention reduces memory to $O(T)$ |

### Architecture summary

```
token IDs x_0, ..., x_T
      ↓ embedding lookup E
sequence of d-dim row vectors
      ↓ (repeated L times)
      ├── RMSNorm
      ├── Multi-Head Causal Attention (Nh heads)
      ├── Residual add
      ├── RMSNorm
      ├── MLP (position-wise, d → 4d → d)
      └── Residual add
      ↓ logit projection (d → |V|)
      ↓ softmax at each position
conditional distributions p_θ(x_t | x_{<t})
```